# പാഠം 05 - ഏജന്റിക് RAG


## സജ്ജീകരണം

ഈ നോട്ട്ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് Agentic RAG (Retrieval-Augmented Generation) മാതൃക കാണിക്കുന്നു.

**പൂര്‍വാവಶ്യങ്ങൾ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — നിങ്ങളുടെ Azure AI Search സർവീസ് എൻഡ്‌പോയിന്റ്
- `AZURE_SEARCH_API_KEY` — നിങ്ങളുടെ Azure AI Search API കീ
- പരിസ്ഥിതി വ്യത്യാസങ്ങളിലൂടെ കോൺഫിഗർ ചെയ്ത Azure OpenAI ഡിപ്പ്ലോയ്മെന്റ്
- Azure CLI ഒത്തുചേർന്നിട്ടുണ്ട് (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ഏജൻറിക് RAG എന്താണ്?

പരമ്പരാഗത RAG ഒരു നിശ്ചിത പൈപ്പ്‌ലൈൻ പിന്തുടരുന്നു: പ്രമാണങ്ങൾ തിരയുക, പിന്നെ ഒരു മറുപടി സൃഷ്ടിക്കുക. **ഏജൻറിക് RAG** കൂടുതൽ ముందേ പോകുന്നു ഏജന്റിന് സ്വാതന്ത്ര്യം നൽകുന്നത് **എപ്പോൾ**യും **എങ്ങനെയെന്ന്** വിവരങ്ങൾ തിരയണമെന്ന് തീരുമാനിക്കാൻ.

ഏജൻറിക് RAG ഉപയോഗിച്ച്, ഏജന്റ്ക്ക് കഴിയും:
- ഒരു ചോദ്യത്തിന് ഉത്തരം നൽകുന്നതിന് മുമ്പ് തിരയൽ ആവശ്യമാണോ എന്ന് **തീർച്ചയാക്കുക**
- ഏത് ഡാറ്റാ ഉറവിടത്തെയോ ടൂളെയോ ചോദിക്കാമെന്ന് **തിരഞ്ഞെടുക്കുക**
- തിരഞ്ഞെടുത്ത ഫലങ്ങൾ **അവലോകനം** ചെയ്ത് ആദ്യ ശ്രമം പോരാത്ത പക്ഷം ഫോളോ-അപ്പ് തിരയലുകൾ നടത്തുക
- പല തിരയൽ ഘട്ടങ്ങളിലെയും വിവരങ്ങൾ സംയോജിപ്പിച്ച് ഏകോപിതമായ ഉത്തരം **സൃഷ്ടിക്കുക**

“一ഴുത്തുപ്രക്രിയയായ തിരയുക-പിന്നെ-സൃഷ്ടിക്കു” എന്നതിനേക്കാൾ ഏജന്റ് കൂടുതൽ സാന്ദ്രവും കൃത്യവുമാകും.


## ഒരു തിരയൽ ടൂൾ സൃഷ്ടിക്കൽ

Agentic RAG-യിൽ, ബാഹ്യ ഡാറ്റാ ഉറവിടങ്ങൾ ഏജൻറ് ആവശ്യത്തിന് വിളിക്കാൻ കഴിയുന്ന **ടൂളുകളായി** പൊതിഞ്ഞിരിക്കുന്നു. ഇത് ഏജന്റിന് ശേഖരണത്തെ മറ്റൊരു നടപടി പോലെ കൈകാര്യം ചെയ്യാൻ സഹായിക്കുന്നു, നിർബന്ധമായ ഘട്ടമല്ലാതെ.

ഇതിൻ്റെ താഴെ ഒരു യാത്രാ അറിവ് അടിസ്ഥാനമായി നിർവ്വചിച്ച്, ഏജൻറ് ലക്ഷ്യസ്ഥലം വിവരങ്ങൾ തേടാൻ വിളിക്കാൻ കഴിയുന്ന ഒരു ടൂൾ ആയി ഇത് പ്രകടിപ്പിക്കുന്നു.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ഏജന്റ് നിർമ്മാണം

ഇപ്പോൾ ഞങ്ങൾ ഒരു ഏജന്റിനെ സൃഷ്‌ടിക്കുന്നു, അത് **എപ്പോഴും മറുപടി നൽകുന്നതിന് മുമ്പ് വിവരങ്ങൾ കണ്ടെത്താൻ നിർദ്ദേശിക്കപ്പെട്ടിരിക്കുന്നു**. ഈ ഏജന്റ് തന്റെ സ്വന്തം പരിശീലന ഡാറ്റയിലും ആശ്രയിക്കാതെ, തന്റെ പ്രതികരണങ്ങൾക്ക് അടിസ്ഥാനമാക്കിയെടുക്കാൻ `search_travel_knowledge` ടൂൾ ഉപയോഗിച്ച് വിജ്ഞാനശേഖരത്തിൽ നിന്നുള്ള അറിവ് ഉപയോഗിക്കുന്നു.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ആവർത്തനാര്ഹമായ തിരയലുകൾ — നിർമ്മാതാവ്-പരിശോധക മാതൃക

Agentic RAG-ന്റെ ഒരു പ്രധാന ലാഭം **ആവർത്തനാര്ഹമായ തിരയലുകൾ** ആണ്. ഏജന്റ് ആരംഭത്തെ കണ്ടെത്തലുകൾ പരിശോധിക്കാനും, മെച്ചപ്പെടുത്താനും, വികസിപ്പിക്കാനും നിരവധി തവണ തിരയലുകൾ നടത്താൻ കഴിയും — "നിർമ്മാതാവ്-പരിശോധക" പ്രവൃത്തി രീതിയെപ്പോലെ:

1. **നിർമ്മാതാവ് ഘട്ടം**: ഏജന്റ് ആരംഭത്തെ വിവരങ്ങൾ ശേഖരിച്ച് ഒരു മറുപടി രൂപപ്പെടുത്തിയെടുക്കുന്നു.
2. **പരിശോധക ഘട്ടം**: ഏജന്റ് വിശദാംശങ്ങൾ സ്ഥിരീകരിക്കാനോ അല്ലെങ്കിൽ വിടവക്കുകൾ പൂരിപ്പിക്കാനോ അധിക തിരയലുകൾ നടത്തുന്നു.

താഴെ, ഏജന്റിന് പല ഗതികളെ താരതമ്യം ചെയ്യേണ്ട ഒരു ചോദ്യം ചോദിക്കപ്പെട്ടിരിക്കുന്നു, അത് പല തവണ തിരയലുകൾ നടത്താൻ പ്രേരിപ്പിക്കുന്നു.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **Agentic RAG** സിസ്റ്റം നിർമ്മിക്കുന്നത് എങ്ങനെ എന്നത് പഠിച്ചു:

- **Agentic RAG** ഏജന്റുകൾ സ്വതന്ത്രമായി റിട്ട്രീവ് ചെയ്യേണ്ടുന്ന സമയം തീരുമാനിക്കാനും, റിട്ട്രീവൽ സ്തിതി സ്ഥിരമായതല്ല, സഞ്ചാരമാക്കാനും സാധിക്കും.
- **ടൂളുകൾ ഡാറ്റാ ഉറവിടങ്ങളായി**: പുറം ലോക തത്വങ്ങൾ (Azure AI Search പോലെയുള്ളവ) ഏജന്റ് വിളിക്കാവുന്ന ടൂളുകളായി പൊക്കപ്പെടുന്നു.
- **പിടിച്ചെടുക്കൽ ആവർത്തനം**: മേക്കർ-ചെക്കാര്ഡ് മാതൃക ഏജന്റിന് ഒന്നിലധികം റിട്ട്രീവൽ റൗണ്ടുകൾ നടത്താൻ അനുവദിക്കുന്നു — തിരയൽ, പരിശോധന, ശുദ്ധീകരണം — ഒടുവിൽ ഒരു തീർത്ഥയായ ഉത്തരം നൽകുന്നതിന് മുമ്പ്.

പ്രോഡക്ഷനിൽ, വലിയ തലത്തിലുള്ള യാത്രാ രേഖകളുടെ റിട്ട്രീവലിനായി മെമ്മറിയിൽ ഉള്ള `TRAVEL_KNOWLEDGE_BASE` യുടെ പകരം യഥാർത്ഥ Azure AI Search ഇൻഡക്സ് ഉപയോഗിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അക്രമപ്പെടുത്തൽ**:  
ഈ ഡോക്യുമെന്റ് AI തർജ്ജമാ സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജ്ജമ ചെയ്‌തതാണ്. അസൂയപരമായ കൃത്യതയതിനായി ശ്രമിച്ചെങ്കിലും, ഓട്ടോമേറ്റഡ് തർജ്ജമകളിൽ പിശകുകൾ അല്ലെങ്കിൽ അസംവിധാനങ്ങൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. നേട്ട കേരള ഭാഷയിലെ അഭിമുഖ ഡോക്യുമെന്റ് അതിന്റെ ശക്തമായ ഉറവിടമായി കണക്കാക്കപ്പെടണം. നിർണായക വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ തർജ്ജമ നിർദ്ദേശിക്കപ്പെടുന്നു. ഈ തർജ്ജമ ഉപയോഗിക്കുന്നതിൽ നിന്നുണ്ടാകുന്ന ബോധമെല്ലാം അല്ലെങ്കിൽ തെറ്റിദ്ധാരണകൾക്കുള്ള ഉത്തരവാദിത്വം ഞങ്ങൾ ഏറ്റെടുക്കരുത്.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
